In [1]:
import sys
sys.path.insert(0, '../')
sys.path.insert(1, '../../')
from build_config import ALL_SEASONS, COMPETITIONS, CURRENT_SEASON, INPUT_CSV_PATHS, TARGET_RANGES
from const import REPO_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, MODELS_PATH
from fbref_const import URLs, TARGET_COLUMNS

from src.data.match_data_processing import process_match_target_var, process_match_other_var
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature

import pandas as pd
import os
import numpy as np
import mlflow
import joblib
import tempfile
import os

from build_utils import *

In [2]:
key_columns=['date', 'home', 'away']

In [3]:
team_encoder_path = f"{MODELS_PATH}/data_processors/{COMPETITIONS[0]}/team_encoder.pkl"
encoder = TeamEncoder.load(team_encoder_path)

In [4]:
seasons=sorted(ALL_SEASONS)

In [5]:
test_dict=dict((k, pd.read_csv(f'{REPO_PATH}/{INPUT_CSV_PATHS[k]}')) for k in COMPETITIONS)

In [6]:
all_features_dict={}
for competition in COMPETITIONS:
    if competition not in test_dict:
        continue
    test_df = test_dict[competition]
    team_encoder_path = f"{MODELS_PATH}/data_processors/{competition}/team_encoder.pkl"
    season_dfs_dict = {season: pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons}
    target_dfs = [pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]
    # TeamEncoder features
    encoder = TeamEncoder.load(team_encoder_path)
    team_encoder_features = encoder.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagFeatureGenerator features
    lag_feature_generator = TeamLagFeatureGenerator(lookback=5)
    team_lag_features = lag_feature_generator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # PreviousSeasonTeamAverager features
    prev_season_averager = PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
    prev_season_features = prev_season_averager.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamRestDaysCalculator features
    rest_days_calculator = TeamRestDaysCalculator()
    rest_days_features = rest_days_calculator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagTargetFeature features
    lag_target_generator = TeamLagTargetFeature(lookback=5)
    lag_target_features = lag_target_generator.transform_spot(season_dfs_dict, target_dfs, test_df).reset_index(drop=True)

    # Merge all features on home, away, date
    from functools import reduce
    feature_dfs = [team_encoder_features, team_lag_features, prev_season_features, rest_days_features, lag_target_features]
    all_features = reduce(lambda left, right: pd.merge(left, right, on=['home', 'away', 'date'], how='outer'), feature_dfs)
    all_features_dict[competition] = all_features.copy()

In [13]:
model_dict={
    'premier_league': {
        'away_goals':{
            'experiment_id': '538763672635562977',
            'run_id': 'b7212d036774402c9fd2c73fe2734048',
            'artifact_path': 'model',
        },
        'home_goals':{
            'experiment_id': '538763672635562977',
            'run_id': '5be8128642e3474cadd59b84f1cbaad9',
            'artifact_path': 'model',
        },
    }
}

In [8]:
def load_joblib_model_from_mlflow(experiment_id, run_id, artifact_path, tracking_uri=None):
    """
    Fetch and load a joblib-dumped model from MLflow given experiment_id, run_id, and artifact_path.
    Optionally specify the MLflow tracking URI.
    Returns the loaded model.
    """
    if tracking_uri is not None:
        mlflow.set_tracking_uri(tracking_uri)
    client = mlflow.tracking.MlflowClient()
    # Download artifact to a temporary directory
    with tempfile.TemporaryDirectory() as tmp_dir:
        local_path = client.download_artifacts(run_id, artifact_path, tmp_dir)
        # Find the first .joblib file in the artifact directory
        for root, _, files in os.walk(local_path):
            for file in files:
                if file.endswith('.joblib'):
                    model_path = os.path.join(root, file)
                    return joblib.load(model_path)
        raise FileNotFoundError("No .joblib model file found in the artifact path.")


In [17]:
predictions={}

In [18]:
for competition in COMPETITIONS:
    predictions[competition] = {}
    for target_name in TARGET_RANGES:
        model = load_joblib_model_from_mlflow(model_dict[competition][target_name]['experiment_id'],
                                                model_dict[competition][target_name]['run_id'],
                                                model_dict[competition][target_name]['artifact_path'],
                                                f'{REPO_PATH}/mlflow')
        prediction = model.predict_proba(all_features_dict[competition].drop(columns=key_columns))
        prediction = pd.DataFrame(prediction, columns=list(range(TARGET_RANGES[target_name][0], TARGET_RANGES[target_name][1]+1))+['other'])
        predictions[competition][target_name]=prediction

In [15]:
# Transform predictions DataFrames: rename columns and update values as specified
for target_name, df in predictions.items():
    cols = df.columns.tolist()
    new_cols = []
    # First column: lte_{original}
    new_cols.append(f"lte_{cols[0]}")
    # Other columns: gt_{previous}
    for i in range(1, len(cols)):
        new_cols.append(f"gt_{cols[i-1]}")
    df.columns = new_cols
    # Transform values: first column stays, others become sum of itself and all to the right
    arr = df.values.copy()
    for i in range(1, arr.shape[1]):
        arr[:, i] = arr[:, i:].sum(axis=1)
    df.iloc[:, :] = arr
    predictions[target_name] = df

# Example: predictions['home_goals'].head()

In [16]:
predictions['home_goals']

,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5
0,0.162236,0.837764,0.675046,0.555970,0.423503,0.245798,0.122217
1,0.168745,0.831254,0.667829,0.547427,0.416401,0.244267,0.121456
2,0.176311,0.823689,0.659621,0.540386,0.422524,0.244583,0.120837
3,0.161444,0.838556,0.688145,0.556807,0.435363,0.252015,0.124509
4,0.169204,0.830796,0.684190,0.554377,0.431212,0.250273,0.124442
5,0.160881,0.839119,0.670244,0.547516,0.420042,0.251748,0.124377
6,0.172838,0.827162,0.678775,0.551232,0.416397,0.250192,0.124402
7,0.176234,0.823766,0.649364,0.523083,0.398255,0.260672,0.129613
8,0.173225,0.826775,0.664977,0.544396,0.412693,0.247343,0.122201
9,0.159441,0.840559,0.675477,0.547797,0.415271,0.248889,0.122964


In [19]:
# Add home, away, date columns from test_dict to each predictions DataFrame
for competition in predictions:
    test_rows = test_dict[competition][['home', 'away', 'date']].reset_index(drop=True)
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        # Prepend home, away, date columns
        df = pd.concat([test_rows, df.reset_index(drop=True)], axis=1)
        predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [21]:
# Rename columns and update values in predictions DataFrames for each competition and target
for competition in predictions:
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        cols = df.columns.tolist()
        # Only rename and update non-metadata columns (assume first three are home, away, date)
        meta_cols = ['home', 'away', 'date']
        feature_cols = cols[3:]
        # Rename columns: first becomes lte_{original}, others become gt_{previous}
        new_cols = meta_cols.copy()
        if feature_cols:
            new_cols.append(f"lte_{feature_cols[0]}")
            for i in range(1, len(feature_cols)):
                new_cols.append(f"gt_{feature_cols[i-1]}")
        # Update values: first feature column stays, others become sum of itself and all to the right
        arr = df[feature_cols].values.copy() if feature_cols else None
        if arr is not None and arr.shape[1] > 0:
            for i in range(1, arr.shape[1]):
                arr[:, i] = arr[:, i:].sum(axis=1)
            df = pd.concat([df[meta_cols].reset_index(drop=True), pd.DataFrame(arr, columns=new_cols[3:])], axis=1)
            df.columns = new_cols
            predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [22]:
predictions['premier_league']['home_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5
0,Manchester Utd,Aston Villa,2025-5-25,0.162236,0.837764,0.675046,0.555970,0.423503,0.245798,0.122217
1,Ipswich Town,West Ham,2025-5-25,0.168745,0.831254,0.667829,0.547427,0.416401,0.244267,0.121456
2,Wolves,Brentford,2025-5-25,0.176311,0.823689,0.659621,0.540386,0.422524,0.244583,0.120837
3,Southampton,Arsenal,2025-5-25,0.161444,0.838556,0.688145,0.556807,0.435363,0.252015,0.124509
4,Liverpool,Crystal Palace,2025-5-25,0.169204,0.830796,0.684190,0.554377,0.431212,0.250273,0.124442
5,Nott'ham Forest,Chelsea,2025-5-25,0.160881,0.839119,0.670244,0.547516,0.420042,0.251748,0.124377
6,Fulham,Manchester City,2025-5-25,0.172838,0.827162,0.678775,0.551232,0.416397,0.250192,0.124402
7,Newcastle Utd,Everton,2025-5-25,0.176234,0.823766,0.649364,0.523083,0.398255,0.260672,0.129613
8,Bournemouth,Leicester City,2025-5-25,0.173225,0.826775,0.664977,0.544396,0.412693,0.247343,0.122201
9,Tottenham,Brighton,2025-5-25,0.159441,0.840559,0.675477,0.547797,0.415271,0.248889,0.122964
